In [1]:
import os
os.chdir("D:\\vector_databse")

In [2]:
from scipy.stats import mode
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core.tools import QueryEngineTool
from dotenv import load_dotenv
load_dotenv()
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0)


In [16]:

# seeting up pinecone
from pinecone import Pinecone, ServerlessSpec
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import SimpleDirectoryReader

In [4]:
pc = Pinecone()

existing = pc.list_indexes()

index_name = []

for index in existing:
    index_name.append(index.name)

print(index_name)

['documid-finance', 'documind-hr', 'documind-product']


In [5]:
index_to_delete = ["documind-hr", "documind-product", "documind-finance"]

for idx_name in index_to_delete:
    if idx_name in index_name:
        # Pass idx_name here instead of index_name
        pc.delete_index(idx_name)
        print(f"Deleted index: {idx_name}")
    else:
        print(f"Index {idx_name} not found")


Deleted index: documind-hr
Deleted index: documind-product
Index documind-finance not found


In [14]:
def get_or_create_index(index_name, pdf_path):
    existing = [i.name for i in pc.list_indexes()]
    
    if index_name not in existing:
        print(f"creating index: {index_name}")
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )

        doc = SimpleDirectoryReader(
            input_files=[pdf_path]
        ).load_data()
        nodes = SentenceSplitter(chunk_size=512, chunk_overlap=64).get_nodes_from_documents(doc)
        
        # store it in pinecone
        vs = PineconeVectorStore(pinecone_index=pc.Index(index_name))
        storage_context = StorageContext.from_defaults(vector_store=vs)
        index = VectorStoreIndex(nodes, storage_context=storage_context)
        print(f"index created: {index_name}")
        return index
        
    else:
        # THE FIX: Load the existing index if it's already in Pinecone!
        print(f"Index '{index_name}' already exists. Loading it...")
        vs = PineconeVectorStore(pinecone_index=pc.Index(index_name))
        index = VectorStoreIndex.from_vector_store(vector_store=vs)
        return index


In [7]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.prompts.default_prompts import DEFAULT_TEXT_QA_PROMPT

In [8]:
from llama_index.core import PromptTemplate

prompt =PromptTemplate("""
You are DocMind - NovaTech's knowledge assistant.
RULES:
- Use ONLY the CONTEXT. No outside knowledge.
- If not in CONTEXT: "Not found in provided documents."
- Max 4 sentences. Add source tag [HR], [PRODUCT], or [FINANCE].
CONTEXT: {context_str}
QUESTION: {query_str}
ANSWER:
""")

In [12]:
hr_index = get_or_create_index("documind-hr", r"dataset/hr_policy_manual.pdf")
product_index = get_or_create_index("documind-product", r"dataset/novacrm_product_manual.pdf")
finance_index = get_or_create_index("documid-finance", r"dataset/annual_financial_report_2024.pdf")

In [15]:
from aiohttp import client_exceptions
hr_tools = QueryEngineTool(
    query_engine = hr_index.as_query_engine(
        text_qa_template=DEFAULT_TEXT_QA_PROMPT,
        similarity_top_k=3,
    ),
    metadata=ToolMetadata(
        name="HR Policy Manual",
        description={
            "use or HR and employee policy question: leave policies"
            "(annual), sick, maternity, paternity, casual, bereavement),"
            "notice period, probation, salary structure, EPF, appraisal," 
            "code of conduct, disciplinary action, WFH policy"       }
    )
)

finance_tool = QueryEngineTool(
    query_engine=finance_index.as_query_engine(
        text_qa_template=DEFAULT_TEXT_QA_PROMPT, similarity_top_k=3

    ),
    metadata=ToolMetadata(
        name="financial_report",
        description= "Use for financial and business performance questions: "
            "revenue, profit, EBITDA, growth, Q1/Q2/Q3/Q4 performance, "
            "risk factors, strategic outlook, international expansion, "
            "employee count, attrition, segment analysis."
        )
    )

product_tool = QueryEngineTool(
    query_engine=product_index.as_query_engine(
        text_qa_template=DEFAULT_TEXT_QA_PROMPT, similarity_top_k=3
    ),
    metadata=ToolMetadata(
        name="product_manual",
        description=(
            "Use for NovaCRM product questions: features, modules, "
            "sales module, customer support, marketing automation, "
            "reporting analytics, integrations, API, pricing tiers, "
            "technical specifications, SLA."
        )
    )
)

AttributeError: 'NoneType' object has no attribute 'as_query_engine'

In [20]:

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0)


pc = pinecone.Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

def get_or_create_index(index_name, pdf_path):
    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=384,
            metric="cosine",
            spec=Pinecone.ServerlessSpec(cloud="aws", region="us-east-1"),
        )

    pinecone_index = pc.Index(index_name)
    vs = PineconeVectorStore(pinecone_index=pinecone_index)
    sc = StorageContext.from_defaults(vector_store=vs)

    doc = SimpleDirectoryReader(input_files=[pdf_path]).load_data()
    nodes = SentenceSplitter(chunk_size=512, chunk_overlap=64).get_nodes_from_documents(doc)

    index = VectorStoreIndex(nodes, storage_context=sc)
    return index

hr_index = get_or_create_index("documind-hr", r"dataset/hr_policy_manual.pdf")
product_index = get_or_create_index("documind-product", r"dataset/novacrm_product_manual.pdf")
finance_index = get_or_create_index("documid-finance", r"dataset/annual_financial_report_2024.pdf")

AttributeError: module 'llama_index.vector_stores.pinecone' has no attribute 'Pinecone'